# 🎙️ Estúdio de Clonagem de Voz (XTTS-v2)
### Pipeline Gratuito com GPU T4 e Geração Ilimitada

Este notebook permite clonar a voz do seu irmão a partir da amostra gravada e gerar narrações completas para qualquer novo roteiro em Português do Brasil.

> 💡 **Como usar:**
> 1. Verifique se o Colab está conectado com GPU: no menu superior, clique em **Ambiente de execução** > **Alterar tipo de ambiente de execução** > Selecione **T4 GPU**.
> 2. Execute a **Célula 1** (instalação das dependências).
> 3. Execute a **Célula 2** (inicialização do modelo e interface Web).
> 4. Abra o link público do **Gradio** (`xxxx.gradio.live`) que aparecerá no final da execução!

## ⚙️ Célula 1: Instalação das Dependências (Leva ~1 a 2 minutos)

In [ ]:
# Instalação do Coqui TTS (fork oficial compatível com Python 3.10-3.14) e Gradio
!pip install --upgrade pip
!pip install -q coqui-tts gradio
print("\n✅ Dependências instaladas com sucesso!")

## 🚀 Célula 2: Inicialização do Modelo e Interface Web

In [ ]:
import os
import torch
import gradio as gr
from TTS.api import TTS

# 1. Verificar aceleração GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔥 Dispositivo ativo: {device.upper()}")
if device != "cuda":
    print("⚠️ ATENÇÃO: Nenhuma GPU detectada. Ative a GPU T4 em: Ambiente de execução > Alterar tipo de ambiente de execução > T4 GPU")

# 2. Carregar modelo XTTS-v2
print("📥 Carregando modelo XTTS-v2 (pode levar 1 minuto no primeiro uso)...")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("✅ Modelo pronto para síntese vocal!")

# 3. Função de Clonagem de Voz
def sintetizar_clone(audio_referencia, texto_roteiro, velocidade, idioma):
    if not audio_referencia:
        return None, "❌ Por favor, faça o upload de um áudio de referência da voz."
    if not texto_roteiro or not texto_roteiro.strip():
        return None, "❌ Por favor, digite o texto do roteiro a ser narrado."
    
    output_file = "audio_clonado_final.wav"
    try:
        tts.tts_to_file(
            text=texto_roteiro.strip(),
            speaker_wav=audio_referencia,
            language=idioma,
            speed=velocidade,
            file_path=output_file
        )
        return output_file, "✅ Narração gerada com sucesso!"
    except Exception as e:
        return None, f"❌ Erro na síntese: {str(e)}"

# 4. Construção da Interface Web (Gradio)
with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue")) as demo:
    gr.Markdown("# 🎙️ Estúdio de Clonagem de Voz (XTTS-v2)")
    gr.Markdown("Suba a amostra de áudio do seu irmão, cole o roteiro e gere áudios clonados com entonação humana e natural.")
    
    with gr.Row():
        with gr.Column(scale=1):
            audio_input = gr.Audio(
                label="1. Áudio de Referência (Voz do seu irmão)",
                type="filepath"
            )
            gr.Markdown("💡 *Dica: Suba o arquivo `amostra_voz_irmao_30s.wav` que geramos.*")
            
            with gr.Accordion("Opções Avançadas", open=False):
                idioma_input = gr.Dropdown(
                    choices=["pt", "en", "es", "fr"],
                    value="pt",
                    label="Idioma da Fala"
                )
                speed_slider = gr.Slider(
                    minimum=0.7,
                    maximum=1.4,
                    value=1.0,
                    step=0.05,
                    label="Velocidade da Narração (1.0 = normal)"
                )
        
        with gr.Column(scale=2):
            text_input = gr.Textbox(
                label="2. Roteiro / Texto a ser narrado",
                placeholder="Cole aqui o texto do novo roteiro que você deseja que ele narre...",
                lines=8
            )
            btn_gerar = gr.Button("🚀 Gerar Narração com a Voz Clonada", variant="primary", size="lg")
            
            status_output = gr.Markdown("")
            audio_output = gr.Audio(label="3. Áudio Final Clonado", type="filepath")
    
    btn_gerar.click(
        fn=sintetizar_clone,
        inputs=[audio_input, text_input, speed_slider, idioma_input],
        outputs=[audio_output, status_output]
    )

# Iniciar com link público seguro (share=True)
demo.launch(share=True, debug=True)
